# Exporting a scene to glTF

K3D can hand the scene over as a binary glTF (`.glb`) - a portable model that opens in Blender, MeshLab or a slicer for 3D printing. Unlike a PNG or an HTML snapshot, what comes out is geometry you can keep working on.

In [ ]:
from base64 import b64decode

import numpy as np

import k3d

In [ ]:
T = 1.618033988749895
r = 4.77
x, y, z = np.meshgrid(*[np.linspace(-r, r, 77)] * 3, indexing='ij')

gyroid = 2 - (np.cos(x + T * y) + np.cos(x - T * y) +
              np.cos(y + T * z) + np.cos(y - T * z) +
              np.cos(z - T * x) + np.cos(z + T * x)).astype(np.float32)

markers = np.random.uniform(-0.5, 0.5, (30, 3)).astype(np.float32)

plot = k3d.plot()
plot += k3d.marching_cubes(gyroid, level=0.0, name='gyroid')
plot += k3d.points(markers, point_size=0.04, shader='mesh', color=0xff2244, name='markers')
plot.display()

## From Python

`fetch_gltf` asks the browser to build the file. The answer travels back over the widget comm, so it lands in `plot.gltf` only after the current cell has finished - run the write in the next one.

In [ ]:
plot.fetch_gltf()

In [ ]:
with open('scene.glb', 'wb') as f:
    f.write(b64decode(plot.gltf))

To keep it in one cell, `yield_gltfs` turns the round trip into a generator that resumes once the file arrives.

In [ ]:
@plot.yield_gltfs
def export():
    plot.fetch_gltf()
    glb = yield

    with open('scene.glb', 'wb') as f:
        f.write(glb)

    print('wrote %.1f MB' % (len(glb) / 1e6))


export()

## What travels and what does not

glTF describes triangles and PBR materials. An object whose shape only exists while its shader runs has nothing to hand over, so it is left out rather than exported as the proxy geometry that shader consumes - a volume would otherwise arrive as a plain cube. The browser console lists whatever was skipped.

**Exported:** `mesh`, `surface`, `stl`, `marching_cubes`, `voxels`, `sparse_voxels`, `voxels_group`, `texture` built from an image, `points(shader='mesh')`, `line` and `lines` with `shader='mesh'` (tubes) or `shader='simple'`, and the arrowheads of `vectors` and `vector_field`.

**Left out:** `volume`, `mip`, `volume_slice`, `texture` built from an `attribute`, `points` with shader `'3d'`, `'dot'` or `'flat'`, `line` and `lines` with `shader='thick'`, `text`, `text2d`, `label`, `texture_text`, and the shafts of `vectors` and `vector_field`.

Switching a points or lines object to `shader='mesh'` is enough to bring it into the export. The grid, axes and lights are not part of the model and never travel.

## Outside the notebook

The **Export glTF** button in the Controls panel does the same work entirely in the browser, so it also works inside an HTML snapshot, where there is no kernel to ask.

For scripts and CI there is a headless route: a plain `.py` file, no notebook and no widget, just a Chrome that K3D starts and drives itself. It needs a working Chrome installation, and unlike `fetch_gltf` it is synchronous - the call returns the bytes.

In [ ]:
from k3d.headless import k3d_remote, get_headless_driver

headless = k3d_remote(plot, get_headless_driver())
headless.sync(hold_until_refreshed=True)

with open('scene_headless.glb', 'wb') as f:
    f.write(headless.get_gltf())

headless.close()